In [3]:
import os
import random
import sys
import warnings
from pathlib import Path
import optuna
import time

import numpy as np
import pandas as pd
import math
import matplotlib.dates as mdates
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.tcn_wrapper import TCNModel
from src.eval.scorer import VolatilityScorer
from src.trainer.walk_forward import WalkForwardTrainer
from src.data.dataset import TimeSeriesDataset
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst
from src.utils import test_toolkit as ts

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "tcn"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any(), "NaNs en train"
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any(), "NaNs en test"
assert df_train.index.max() < df_test.index.min(), "Solapamiento temporal train/test"

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"¡GPU Detectada! Usando: {torch.cuda.get_device_name(0)}")
else:
    print("ALERTA: No se detectó GPU")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)
ALERTA: No se detectó GPU


## Introducción

La Red Convolucional Temporal (*Temporal Convolutional Network*, TCN) es una arquitectura diseñada para modelar dependencias temporales en datos secuenciales. A diferencia de las redes neuronales tradicionales, incorpora mecanismos de convolución que permiten capturar patrones de largo plazo expandiendo el campo receptivo de forma exponencial mediante dilataciones, sin sufrir el desvanecimiento del gradiente.

En el contexto de las series financieras, esta arquitectura ofrece una alternativa a los modelos econométricos tradicionales, al permitir que la relación entre los retornos históricos y la volatilidad futura sea aprendida directamente a partir de los datos. Su estructura basada en convoluciones causales facilita la identificación de patrones temporales y cambios en la dinámica de los retornos, sin establecer previamente una ecuación paramétrica para la evolución de la varianza condicional.

El objetivo de esta sección es presentar, entrenar y evaluar un modelo TCN para la predicción de la volatilidad del crudo Brent bajo el protocolo del benchmark. Se estudia el efecto de distintas configuraciones de convolución (tamaño del kernel, número de niveles, número de convoluciones por bloque y dilatación) sobre el desempeño predictivo de la red, manteniendo las condiciones de información y evaluación establecidas para los demás modelos del proyecto.

### Arquitectura

La arquitectura TCN se fundamenta en la combinación de tres componentes principales: convoluciones causales unidimensionales, convoluciones dilatadas y bloques residuales. A diferencia de las redes recurrentes que procesan la secuencia paso a paso, la TCN aplica filtros convolucionales en paralelo a lo largo de toda la ventana temporal, garantizando que la información fluya únicamente desde el pasado hacia el futuro.

Para una observación de entrada $\mathbf{x}_t$ y un filtro $\mathbf{w}$ de tamaño $K$, la operación de convolución causal, que evita el sesgo de anticipación (*look-ahead bias*), se expresa como:

$$
(\mathbf{x} * \mathbf{w})[t] = \sum_{k=0}^{K-1} \mathbf{w}[k] \cdot \mathbf{x}[t - k].
$$

Para capturar dependencias de largo plazo sin aumentar excesivamente el número de parámetros, se incorporaron **convoluciones dilatadas**. La dilatación introduce un espacio entre los elementos del filtro, permitiendo que el campo receptivo crezca exponencialmente con la profundidad. Para un nivel $l$, el factor de dilatación se define como $d = 2^l$. La operación resultante es:

$$
(\mathbf{x} *_{d} \mathbf{w})[t] = \sum_{k=0}^{K-1} \mathbf{w}[k] \cdot \mathbf{x}[t - d \cdot k].
$$

Para estabilizar el entrenamiento en redes profundas, la TCN se compone de **bloques residuales**. Cada bloque contiene una o más convoluciones causales dilatadas seguidas de activaciones ReLU y capas de *Dropout*. La salida se suma a su entrada mediante una conexión residual (*skip connection*):

$$
\mathbf{h}^{(l)} = \text{ReLU}\left( \mathcal{F}\left(\mathbf{h}^{(l-1)}\right) + \mathbf{h}^{(l-1)} \right),
$$

donde $\mathcal{F}(\cdot)$ representa la transformación convolucional y $\mathbf{h}^{(l)}$ el mapa de características en la capa $l$. Si las dimensiones de entrada y salida difieren, se aplica una convolución $1 \times 1$ en la conexión residual para igualar los canales.

Una vez procesada la secuencia, se extrae el último vector temporal del mapa de características, $\mathbf{h}_t \in \mathbb{R}^{C}$ (donde $C$ es el número de canales de la última capa). Para generar un pronóstico escalar de la varianza condicional, este vector se proyecta a través de una capa densa con pesos $\mathbf{w} \in \mathbb{R}^{C}$ y sesgo $b \in \mathbb{R}$:

$$
y_t = \mathbf{w}^{\top}\mathbf{h}_t + b.
$$

Para garantizar que la varianza pronosticada sea estrictamente positiva, el resultado $y_t$ pasa por una función de activación *Softplus* junto con un término de estabilidad $\varepsilon$:

$$
\hat{\sigma}_t^2 = \text{Softplus}(y_t) + \varepsilon = \ln(1 + e^{y_t}) + \varepsilon,
$$

donde $\boldsymbol{\theta}$ agrupa los pesos y sesgos tanto de las capas convolucionales como de la capa de salida.

Siguiendo el mismo esquema de preprocesamiento empleado en las arquitecturas recurrentes (GRU y LSTM), los retornos logarítmicos diarios $r_t$ fueron estandarizados mediante una transformación Z-score para evitar el desvanecimiento de los gradientes. Para prevenir la fuga de información, el escalador se ajustó exclusivamente con el conjunto de entrenamiento de cada fold. Sea $\tilde{r}_t$ el retorno estandarizado, para una ventana de longitud $L$, la secuencia de entrada se definió como $\mathbf{Z}_t \in \mathbb{R}^{L\times 1}$. 

El problema se abordó mediante una estrategia de predicción de un paso hacia adelante (*one-step-ahead forecasting*). El entrenamiento consistió en encontrar los parámetros $\boldsymbol{\theta}$ que minimizaran la pérdida QLIKE sobre las observaciones disponibles, dado que esta métrica penaliza asimétricamente las subestimaciones de la volatilidad:

$$
\boldsymbol{\theta}^{*}
=\arg\min_{\boldsymbol{\theta}}
\frac{1}{N}
\sum_{i=1}^{N}
\mathcal{L}_{\text{QLIKE}}
\left(
RS_{i},
\hat{RS}_{i}
\right),
$$

donde $N$ es el número de secuencias de entrenamiento y $RS_i$ es la varianza de Rogers-Satchell utilizada como *target*. 

Los parámetros se actualizaron mediante retropropagación del error estándar (*backpropagation*) y el optimizador Adaptive Moment Estimation (Adam). La elección del tamaño del kernel $\left(K\right)$, el número de niveles de dilatación, el número de canales ocultos $\left(C\right)$ y la longitud de las secuencias $\left(L\right)$ influyó directamente sobre el campo receptivo y la capacidad de generalización del modelo. La implementación se realizó utilizando la librería `PyTorch` en su versión `2.13.0`, aprovechando su módulo de diferenciación automática.

## Entrenamiento

Para la selección de hiperparámetros del modelo TCN, se mantuvo el mismo protocolo experimental (Walk-Forward) y el mismo presupuesto computacional que en los modelos paramétricos y recurrentes. A diferencia del GARCH y el EGARCH, la búsqueda de Optuna exploró la arquitectura de la red convolucional y sus parámetros de regularización.

Para adaptar la serie temporal al formato requerido por la red, se construyó una matriz de rezagos (*lag matrix*) para cada ventana del esquema Walk-Forward. Siguiendo los principios de validación cruzada para series temporales (`timeseriescv`), cada fila de esta matriz contenía una secuencia histórica de longitud `look_back`, la cual se utilizó como vector de entrada para predecir el paso inmediatamente posterior. Este enfoque garantizó la integridad temporal de los datos y evitó cualquier tipo de fuga de información futura (*look-ahead bias*) durante el entrenamiento.

El espacio de búsqueda incluyó la longitud de la ventana de entrada (`look_back`), el tamaño del kernel (`kernel_size`), el número de convoluciones por bloque (`num_convs`), el número de canales ocultos (`channels`), la tasa de abandono (`dropout`) y la tasa de aprendizaje. Para evitar que la red procesara secuencias sin suficiente contexto temporal, el número de niveles de dilatación (`num_levels`) no se dejó completamente al azar: se calculó un límite inferior matemático utilizando la fórmula del campo receptivo, garantizando que la capa final pudiera "ver" la totalidad del `look_back` seleccionado. Para mitigar el sobreajuste, se incorporó también la penalización $\ell_2$ (`weight_decay`). 

El entrenamiento se configuró con un máximo de 200 épocas y una paciencia de 20 para la parada temprana interna de la red. El desempeño se siguió evaluando mediante la pérdida QLIKE sobre los conjuntos de validación.

In [ ]:
def tcn_search_space(trial: optuna.Trial) -> dict:
    look_back = trial.suggest_categorical("look_back", [7, 14, 21, 28])
    kernel_size = trial.suggest_categorical("kernel_size", [2, 3])
    num_convs = trial.suggest_categorical("num_convs", [1, 2, 3])

    min_levels = math.ceil(
        math.log2(1 + (look_back - 1) / (num_convs * (kernel_size - 1)))
    )
    max_levels = min(min_levels + 2, 5)

    return {
        "look_back": look_back,
        "kernel_size": kernel_size,
        "num_convs": num_convs,
        "num_levels": trial.suggest_int("num_levels", min_levels, max_levels),
        "channels": trial.suggest_categorical("channels", [32, 64, 128]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "max_epochs": 200,
        "patience": 20,
        "retrain_freq": 1,
        "seed": 42
    }

exec_id = 6
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=TCNModel,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5, warmup=current_warmup,
    embargo=1, n_trials=50,
    suggest_params_fn=tcn_search_space,
    early_stop_rounds=15
)

trainer.run(y_train, rs_train)

El algoritmo de búsqueda se detuvo en el trial 19 sin alcanzar el límite máximo de 50 evaluaciones, debido a la activación del mecanismo de parada temprana (*Early Stopping*) tras 15 intentos consecutivos sin mejoras. El mecanismo de poda (*Pruning*) no descartó ninguna configuración (0 podados). Esta ausencia de podas contrastó con el GRU (que tuvo 6 podas), evidenciando que la TCN, al procesar las secuencias de forma paralela y no sufrir el desvanecimiento de gradiente típico de las redes recurrentes, presentó una convergencia más robusta en las primeras épocas. La mejor configuración registró un QLIKE de 0.4370.

El ajuste de los pesos bajo el esquema Walk-Forward tuvo una duración total de 5.98 minutos (359.10 segundos), con un tiempo medio de 3.78 segundos por pliegue. Es fundamental precisar que este costo corresponde exclusivamente al entrenamiento de la red. La evaluación completa del modelo, que incluyó el reentrenamiento iterativo durante la generación de pronósticos (*forecast*) para las observaciones fuera de muestra, elevó el tiempo de entrenamiento total a aproximadamente 12 horas y 50 minutos.

In [5]:
resumen_data = []

exec_id = 4
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_qlike = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(study_name="TCNModel_study", storage=storage_url)

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])

        mejor_qlike = round(study.best_value, 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Mejor QLIKE": mejor_qlike,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,Mejor QLIKE,Tiempo Total (s),Tiempo por Fold (s)
19,19,0,0.4370,359.1000,3.7800


El ranking de configuraciones mostró que las cinco mejores opciones presentaron una pérdida QLIKE en el rango de 0.437 a 0.488. Se observó un consenso estricto en el uso de un tamaño de kernel de 3 ($K=3$), lo que indicó que la red encontró patrones locales de tres días altamente informativos. A diferencia del GRU (que convergió hacia ventanas de 7 días y una capa), la TCN requirió ventanas más largas (14 y 21 días) y mayor profundidad estructural (4 y 5 niveles de dilatación). Al carecer de un estado oculto recurrente, la TCN pudo haber compensado esta limitación expandiendo su campo receptivo mediante las convoluciones dilatadas.

In [9]:
exec_dir = RESULTS_PATH / "exec_4"

df_tcn = pd.read_csv(exec_dir / "TCNModel_results.csv")

df_tcn["lr"] = df_tcn["learning_rate"].apply(lambda x: f"{x:.4f}")
df_tcn["wd"] = df_tcn["weight_decay"].apply(lambda x: f"{x:.1e}")

cols_agrupar = ["look_back", "kernel_size", "num_convs", "num_levels", "channels", "dropout", "lr", "wd"]
df_agg = df_tcn.groupby(cols_agrupar)["qlike"].agg(["mean", "std"]).reset_index()
df_agg["QLIKE"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)
df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "look_back": "Lookback",
    "kernel_size": "Kernel",
    "num_convs": "Convs",
    "num_levels": "Niveles",
    "channels": "Canales",
    "dropout": "Dropout",
    "lr": "LR",
    "wd": "Weight Decay"
})

df_final = df_top5[["Lookback", "Kernel", "Convs", "Niveles", "Canales", "Dropout", "LR", "Weight Decay", "QLIKE"]]

tt.display_styled_df(df=df_final, align_left_first=False, decimals=3)

Lookback,Kernel,Convs,Niveles,Canales,Dropout,LR,Weight Decay,QLIKE
14,3,2,5,64,0.232,0.0072,6.0e-04,0.437 ± 0.111
21,3,1,5,128,0.235,0.0087,7.2e-04,0.454 ± 0.148
21,3,3,4,32,0.266,0.0071,4.0e-06,0.464 ± 0.154
21,3,1,5,128,0.231,0.0097,4.3e-04,0.464 ± 0.176
14,3,2,4,64,0.228,0.0046,3.4e-04,0.488 ± 0.221


Las dos configuraciones con menor pérdida fueron: (14 días, 64 canales) con un QLIKE de 0.437, y (21 días, 128 canales) con 0.454. Dado que la diferencia entre ambas (0.017) fue menor que la variabilidad observada entre los folds ($\sigma \approx 0.11$ a $0.15$), fueron estadísticamente indistinguibles en desempeño predictivo. No obstante, la desviación estándar del QLIKE en la TCN ($\sigma \approx 0.11$) fue inferior a la del GRU ($\sigma \approx 0.14$), atribuido al procesamiento paralelo de las convoluciones que evita el arrastre secuencial del gradiente típico de las redes recurrentes. Ante la equivalencia en la media, se seleccionó la primera configuración al registrar el mínimo absoluto de QLIKE y requerir la mitad de canales (64 frente a 128), aplicando un criterio de parsimonia que reduce la complejidad paramétrica sin sacrificar capacidad predictiva.